# 3. Product & Growth Metrics

This notebook transforms the validated analytical outputs from the data-processing stage into product and growth metrics.

The analysis focuses on understanding user engagement, product interaction, conversion behaviour, and overall growth performance.

The metrics created here are designed to support downstream segmentation, business analysis, visualization, and the Product & Growth Intelligence application.

## Objectives

- Measure user and session engagement
- Quantify product interaction and activity
- Analyze behavioral funnel progression
- Establish conversion-oriented metrics
- Create reusable metric tables for downstream analysis
- Preserve the validated processing decisions from the previous notebook

No additional raw-data cleaning or transformation is performed in this notebook.

### 3.1 Load Validated Analytical Outputs

The processed analytical datasets created and validated in `02_data_processing.ipynb` are loaded from the processed-data layer.

This notebook uses these persisted outputs as the starting point for product and growth metric computation.

No re-cleaning or re-processing of the raw data is performed here.

In [1]:
# Load validated analytical outputs from the processing stage

from pathlib import Path
import pandas as pd

processed_dir = Path(
    r"D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed"
)

events_clean = pd.read_csv(processed_dir / "events_clean.csv")
session_summary = pd.read_csv(processed_dir / "session_summary.csv")
visitor_summary = pd.read_csv(processed_dir / "visitor_summary.csv")
events_enriched = pd.read_csv(processed_dir / "events_enriched.csv")
product_category_enriched = pd.read_csv(
    processed_dir / "product_category_enriched.csv"
)
category_reference = pd.read_csv(processed_dir / "category_reference.csv")

print("Validated analytical outputs loaded successfully.")

Validated analytical outputs loaded successfully.


### 3.2 Analytical Output Inventory

The validated datasets loaded from the processing stage are inspected to confirm their available analytical dimensions before metric construction.

This check confirms the available row counts and columns without modifying the datasets.

In [2]:
# Inspect the loaded analytical datasets

analytical_outputs = {
    "events_clean": events_clean.shape,
    "session_summary": session_summary.shape,
    "visitor_summary": visitor_summary.shape,
    "events_enriched": events_enriched.shape,
    "product_category_enriched": product_category_enriched.shape,
    "category_reference": category_reference.shape,
}

analytical_outputs

{'events_clean': (2755641, 15),
 'session_summary': (1761675, 12),
 'visitor_summary': (1407580, 18),
 'events_enriched': (2755641, 16),
 'product_category_enriched': (416921, 4),
 'category_reference': (1669, 3)}

### 3.3 Session Engagement Metrics

Session-level engagement metrics are created from the validated session summary data.

The metrics capture session activity and engagement intensity while preserving the session-level grain established during data processing.

In [3]:
# Inspect session-level fields available for metric construction

session_summary.columns.tolist()

['visitorid',
 'session_id',
 'session_start',
 'session_end',
 'session_duration_minutes',
 'event_count',
 'unique_products',
 'views',
 'add_to_carts',
 'transaction_events',
 'transactions',
 'has_transaction']

### 3.3.1 Session Engagement Intensity

Session engagement intensity is measured using activity relative to session duration.

The derived metrics capture how actively users interact with the product during each session while retaining the original session-level grain.

In [4]:
# Create session-level engagement intensity metrics

session_metrics = session_summary.copy()

# Avoid division by zero for sessions with zero duration
session_metrics["events_per_minute"] = (
    session_metrics["event_count"]
    / session_metrics["session_duration_minutes"].replace(0, pd.NA)
)

session_metrics["products_per_minute"] = (
    session_metrics["unique_products"]
    / session_metrics["session_duration_minutes"].replace(0, pd.NA)
)

session_metrics["views_per_minute"] = (
    session_metrics["views"]
    / session_metrics["session_duration_minutes"].replace(0, pd.NA)
)

session_metrics["carts_per_minute"] = (
    session_metrics["add_to_carts"]
    / session_metrics["session_duration_minutes"].replace(0, pd.NA)
)

session_metrics.head()

,visitorid,session_id,session_start,session_end,session_duration_minutes,event_count,unique_products,views,add_to_carts,transaction_events,transactions,has_transaction,events_per_minute,products_per_minute,views_per_minute,carts_per_minute
0,0,0_1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,5.462267,3,3,3,0,0,0,0,0.549223,0.549223,0.549223,0.0
1,1,1_1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,0.000000,1,1,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>
2,2,2_1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,29.221300,8,4,8,0,0,0,0,0.273773,0.136886,0.273773,0.0
3,3,3_1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,0.000000,1,1,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>
4,4,4_1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,0.000000,1,1,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>


### 3.4 Product Interaction Metrics

Product interaction metrics quantify how users explore and interact with products during each session.

These metrics use the validated product-related session fields and preserve the session-level analytical grain.

In [5]:
# Create product interaction metrics at session level

session_metrics["products_per_event"] = (
    session_metrics["unique_products"]
    / session_metrics["event_count"].replace(0, pd.NA)
)

session_metrics["views_per_product"] = (
    session_metrics["views"]
    / session_metrics["unique_products"].replace(0, pd.NA)
)

session_metrics["cart_rate"] = (
    session_metrics["add_to_carts"]
    / session_metrics["views"].replace(0, pd.NA)
)

session_metrics[
    [
        "session_id",
        "unique_products",
        "views",
        "add_to_carts",
        "products_per_event",
        "views_per_product",
        "cart_rate"
    ]
].head()

,session_id,unique_products,views,add_to_carts,products_per_event,views_per_product,cart_rate
0,0_1,3,3,0,1.0,1.0,0.0
1,1_1,1,1,0,1.0,1.0,0.0
2,2_1,4,8,0,0.5,2.0,0.0
3,3_1,1,1,0,1.0,1.0,0.0
4,4_1,1,1,0,1.0,1.0,0.0


### 3.5 Conversion Metrics

Conversion metrics quantify the progression from product interaction toward transaction activity at the session level.

These metrics provide reusable measures of cart activity and transaction conversion without altering the validated source datasets.

In [6]:
# Create session-level conversion metrics

session_metrics["transaction_rate"] = (
    session_metrics["transactions"]
    / session_metrics["event_count"].replace(0, pd.NA)
)

session_metrics["cart_to_transaction_rate"] = (
    session_metrics["transactions"]
    / session_metrics["add_to_carts"].replace(0, pd.NA)
)

session_metrics["transaction_event_rate"] = (
    session_metrics["transaction_events"]
    / session_metrics["event_count"].replace(0, pd.NA)
)

session_metrics[
    [
        "session_id",
        "add_to_carts",
        "transaction_events",
        "transactions",
        "has_transaction",
        "transaction_rate",
        "cart_to_transaction_rate",
        "transaction_event_rate"
    ]
].head()

,session_id,add_to_carts,transaction_events,transactions,has_transaction,transaction_rate,cart_to_transaction_rate,transaction_event_rate
0,0_1,0,0,0,0,0.0,<NA>,0.0
1,1_1,0,0,0,0,0.0,<NA>,0.0
2,2_1,0,0,0,0,0.0,<NA>,0.0
3,3_1,0,0,0,0,0.0,<NA>,0.0
4,4_1,0,0,0,0,0.0,<NA>,0.0


### 3.6 Visitor Engagement Metrics

Visitor-level metrics summarize engagement across sessions and provide a user-level view of product interaction and conversion behaviour.

These metrics are derived from the validated visitor summary data and preserve the visitor-level grain established during data processing.

In [7]:
# Inspect visitor-level fields available for metric construction

visitor_summary.columns.tolist()

['visitorid',
 'total_sessions',
 'first_activity',
 'last_activity',
 'total_events',
 'total_views',
 'total_add_to_carts',
 'total_transaction_events',
 'total_transactions',
 'converted_sessions',
 'total_session_duration_minutes',
 'average_session_duration_minutes',
 'active_days',
 'events_per_session',
 'views_per_session',
 'cart_rate',
 'transaction_rate',
 'session_conversion_rate']

### 3.6 Visitor Engagement Metrics

Visitor-level engagement metrics summarize how users interact with the product across their sessions.

These metrics extend the validated visitor summary with normalized measures of activity intensity while preserving the visitor-level grain.

In [8]:
# Create visitor-level engagement metrics

visitor_metrics = visitor_summary.copy()

visitor_metrics["sessions_per_active_day"] = (
    visitor_metrics["total_sessions"]
    / visitor_metrics["active_days"].replace(0, pd.NA)
)

visitor_metrics["events_per_active_day"] = (
    visitor_metrics["total_events"]
    / visitor_metrics["active_days"].replace(0, pd.NA)
)

visitor_metrics["views_per_active_day"] = (
    visitor_metrics["total_views"]
    / visitor_metrics["active_days"].replace(0, pd.NA)
)

visitor_metrics["carts_per_session"] = (
    visitor_metrics["total_add_to_carts"]
    / visitor_metrics["total_sessions"].replace(0, pd.NA)
)

visitor_metrics["transactions_per_session"] = (
    visitor_metrics["total_transactions"]
    / visitor_metrics["total_sessions"].replace(0, pd.NA)
)

visitor_metrics.head()

,visitorid,total_sessions,first_activity,last_activity,total_events,total_views,total_add_to_carts,total_transaction_events,total_transactions,converted_sessions,...,events_per_session,views_per_session,cart_rate,transaction_rate,session_conversion_rate,sessions_per_active_day,events_per_active_day,views_per_active_day,carts_per_session,transactions_per_session
0,0,1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,3,0,0,0,0,...,3.0,3.0,0.0,0.0,0.0,1.0,3.0,3.0,0.0,0.0
1,1,1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,1,1,0,0,0,0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
2,2,1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,8,8,0,0,0,0,...,8.0,8.0,0.0,0.0,0.0,1.0,8.0,8.0,0.0,0.0
3,3,1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,1,1,0,0,0,0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,4,1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,1,1,0,0,0,0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0


In [9]:
# Validate visitor-level metric construction

visitor_metric_validation = {
    "visitor_summary_rows": len(visitor_summary),
    "visitor_metrics_rows": len(visitor_metrics),
    "row_count_preserved": len(visitor_summary) == len(visitor_metrics),
    "visitor_ids_unique": visitor_metrics["visitorid"].is_unique,
    "new_metric_nulls": visitor_metrics[
        [
            "sessions_per_active_day",
            "events_per_active_day",
            "views_per_active_day",
            "carts_per_session",
            "transactions_per_session"
        ]
    ].isna().sum().to_dict()
}

visitor_metric_validation

{'visitor_summary_rows': 1407580,
 'visitor_metrics_rows': 1407580,
 'row_count_preserved': True,
 'visitor_ids_unique': True,
 'new_metric_nulls': {'sessions_per_active_day': 0,
  'events_per_active_day': 0,
  'views_per_active_day': 0,
  'carts_per_session': 0,
  'transactions_per_session': 0}}

### 3.7 Product Interaction Metrics

Product-level interaction metrics are derived from the validated event-level product and category data.

These metrics quantify product visibility and interaction intensity while preserving the event-level grain established during data processing.

In [10]:
# Inspect product interaction fields available for metric construction

events_enriched.columns.tolist()

['timestamp',
 'visitorid',
 'event',
 'itemid',
 'transactionid',
 'datetime',
 'date',
 'year',
 'month',
 'week',
 'day_of_week',
 'hour',
 'is_view',
 'is_addtocart',
 'is_transaction',
 'categoryid']

In [11]:
# Create event-level product interaction metrics

product_metrics = events_enriched.copy()

product_metrics["product_interaction_count"] = (
    product_metrics["is_view"]
    + product_metrics["is_addtocart"]
    + product_metrics["is_transaction"]
)

product_metrics["has_product_interaction"] = (
    product_metrics["product_interaction_count"] > 0
)

product_metrics[
    [
        "itemid",
        "event",
        "is_view",
        "is_addtocart",
        "is_transaction",
        "product_interaction_count",
        "has_product_interaction"
    ]
].head()

,itemid,event,is_view,is_addtocart,is_transaction,product_interaction_count,has_product_interaction
0,355908,view,1,0,0,1,True
1,248676,view,1,0,0,1,True
2,318965,view,1,0,0,1,True
3,253185,view,1,0,0,1,True
4,367447,view,1,0,0,1,True


In [12]:
# Validate product interaction metric construction

product_metric_validation = {
    "events_enriched_rows": len(events_enriched),
    "product_metrics_rows": len(product_metrics),
    "row_count_preserved": len(events_enriched) == len(product_metrics),
    "product_interaction_count_nulls": product_metrics[
        "product_interaction_count"
    ].isna().sum(),
    "has_product_interaction_nulls": product_metrics[
        "has_product_interaction"
    ].isna().sum()
}

product_metric_validation

{'events_enriched_rows': 2755641,
 'product_metrics_rows': 2755641,
 'row_count_preserved': True,
 'product_interaction_count_nulls': np.int64(0),
 'has_product_interaction_nulls': np.int64(0)}

In [13]:
# Aggregate product interactions to product level

product_summary = (
    product_metrics
    .groupby("itemid", as_index=False)
    .agg(
        total_interactions=("product_interaction_count", "sum"),
        total_views=("is_view", "sum"),
        total_add_to_carts=("is_addtocart", "sum"),
        total_transactions=("is_transaction", "sum"),
        interaction_events=("has_product_interaction", "sum")
    )
)

product_summary.head()

,itemid,total_interactions,total_views,total_add_to_carts,total_transactions,interaction_events
0,3,2,2,0,0,2
1,4,3,3,0,0,3
2,6,29,29,0,0,29
3,9,2,2,0,0,2
4,15,22,18,3,1,22


In [14]:
# Validate product-level aggregation

product_summary_validation = {
    "source_event_rows": len(product_metrics),
    "product_summary_rows": len(product_summary),
    "unique_product_ids": product_summary["itemid"].nunique(),
    "row_count_matches_unique_products": (
        len(product_summary) == product_summary["itemid"].nunique()
    ),
    "product_id_nulls": product_summary["itemid"].isna().sum(),
    "total_interaction_count": product_summary["total_interactions"].sum(),
    "total_source_interaction_count": product_metrics["product_interaction_count"].sum()
}

product_summary_validation

{'source_event_rows': 2755641,
 'product_summary_rows': 235061,
 'unique_product_ids': 235061,
 'row_count_matches_unique_products': True,
 'product_id_nulls': np.int64(0),
 'total_interaction_count': np.int64(2755641),
 'total_source_interaction_count': np.int64(2755641)}

In [15]:
# Create product-level performance and conversion metrics

product_summary["cart_rate"] = (
    product_summary["total_add_to_carts"]
    / product_summary["total_views"].replace(0, pd.NA)
)

product_summary["transaction_rate"] = (
    product_summary["total_transactions"]
    / product_summary["total_views"].replace(0, pd.NA)
)

product_summary["cart_to_transaction_rate"] = (
    product_summary["total_transactions"]
    / product_summary["total_add_to_carts"].replace(0, pd.NA)
)

product_summary["transaction_per_interaction"] = (
    product_summary["total_transactions"]
    / product_summary["total_interactions"].replace(0, pd.NA)
)

product_summary.head()

,itemid,total_interactions,total_views,total_add_to_carts,total_transactions,interaction_events,cart_rate,transaction_rate,cart_to_transaction_rate,transaction_per_interaction
0,3,2,2,0,0,2,0.0,0.0,<NA>,0.000000
1,4,3,3,0,0,3,0.0,0.0,<NA>,0.000000
2,6,29,29,0,0,29,0.0,0.0,<NA>,0.000000
3,9,2,2,0,0,2,0.0,0.0,<NA>,0.000000
4,15,22,18,3,1,22,0.166667,0.055556,0.333333,0.045455


In [16]:
# Validate product-level performance metrics

product_rate_validation = {
    "product_rows": len(product_summary),
    "cart_rate_nulls": product_summary["cart_rate"].isna().sum(),
    "transaction_rate_nulls": product_summary["transaction_rate"].isna().sum(),
    "cart_to_transaction_rate_nulls": product_summary["cart_to_transaction_rate"].isna().sum(),
    "transaction_per_interaction_nulls": product_summary["transaction_per_interaction"].isna().sum(),
    "cart_rate_above_one": (product_summary["cart_rate"] > 1).sum(),
    "transaction_rate_above_one": (product_summary["transaction_rate"] > 1).sum(),
    "cart_to_transaction_rate_above_one": (
        product_summary["cart_to_transaction_rate"] > 1
    ).sum(),
    "transaction_per_interaction_above_one": (
        product_summary["transaction_per_interaction"] > 1
    ).sum()
}

product_rate_validation

{'product_rows': 235061,
 'cart_rate_nulls': np.int64(223),
 'transaction_rate_nulls': np.int64(223),
 'cart_to_transaction_rate_nulls': np.int64(211158),
 'transaction_per_interaction_nulls': np.int64(0),
 'cart_rate_above_one': np.int64(83),
 'transaction_rate_above_one': np.int64(5),
 'cart_to_transaction_rate_above_one': np.int64(220),
 'transaction_per_interaction_above_one': np.int64(0)}

In [17]:
# Build visitor-based product funnel metrics

product_visitor_funnel = (
    events_enriched[
        [
            "visitorid",
            "itemid",
            "is_view",
            "is_addtocart",
            "is_transaction"
        ]
    ]
    .groupby(["itemid", "visitorid"], as_index=False)
    .max()
    .groupby("itemid", as_index=False)
    .agg(
        unique_view_visitors=("is_view", "sum"),
        unique_cart_visitors=("is_addtocart", "sum"),
        unique_transaction_visitors=("is_transaction", "sum")
    )
)

# Remove the event-count-based rate definitions
product_summary = product_summary.drop(
    columns=[
        "cart_rate",
        "transaction_rate",
        "cart_to_transaction_rate"
    ]
)

# Add visitor-based funnel metrics
product_summary = product_summary.merge(
    product_visitor_funnel,
    on="itemid",
    how="left"
)

product_summary["cart_conversion_rate"] = (
    product_summary["unique_cart_visitors"]
    / product_summary["unique_view_visitors"].replace(0, pd.NA)
)

product_summary["transaction_conversion_rate"] = (
    product_summary["unique_transaction_visitors"]
    / product_summary["unique_view_visitors"].replace(0, pd.NA)
)

product_summary["cart_to_transaction_rate"] = (
    product_summary["unique_transaction_visitors"]
    / product_summary["unique_cart_visitors"].replace(0, pd.NA)
)

product_summary.head()

,itemid,total_interactions,total_views,total_add_to_carts,total_transactions,interaction_events,transaction_per_interaction,unique_view_visitors,unique_cart_visitors,unique_transaction_visitors,cart_conversion_rate,transaction_conversion_rate,cart_to_transaction_rate
0,3,2,2,0,0,2,0.000000,2,0,0,0.0,0.0,<NA>
1,4,3,3,0,0,3,0.000000,3,0,0,0.0,0.0,<NA>
2,6,29,29,0,0,29,0.000000,26,0,0,0.0,0.0,<NA>
3,9,2,2,0,0,2,0.000000,2,0,0,0.0,0.0,<NA>
4,15,22,18,3,1,22,0.045455,13,2,1,0.153846,0.076923,0.5


In [18]:
# Validate visitor-based product funnel metrics

product_funnel_validation = {
    "product_rows": len(product_summary),
    "product_ids_unique": product_summary["itemid"].is_unique,
    "cart_conversion_rate_nulls": product_summary[
        "cart_conversion_rate"
    ].isna().sum(),
    "transaction_conversion_rate_nulls": product_summary[
        "transaction_conversion_rate"
    ].isna().sum(),
    "cart_to_transaction_rate_nulls": product_summary[
        "cart_to_transaction_rate"
    ].isna().sum(),
    "cart_conversion_rate_above_one": (
        product_summary["cart_conversion_rate"] > 1
    ).sum(),
    "transaction_conversion_rate_above_one": (
        product_summary["transaction_conversion_rate"] > 1
    ).sum(),
    "cart_to_transaction_rate_above_one": (
        product_summary["cart_to_transaction_rate"] > 1
    ).sum()
}

product_funnel_validation

{'product_rows': 235061,
 'product_ids_unique': True,
 'cart_conversion_rate_nulls': np.int64(223),
 'transaction_conversion_rate_nulls': np.int64(223),
 'cart_to_transaction_rate_nulls': np.int64(211158),
 'cart_conversion_rate_above_one': np.int64(45),
 'transaction_conversion_rate_above_one': np.int64(4),
 'cart_to_transaction_rate_above_one': np.int64(152)}

In [19]:
# Inspect products with conversion rates above 1

problem_products = product_summary[
    (product_summary["cart_conversion_rate"] > 1) |
    (product_summary["transaction_conversion_rate"] > 1) |
    (product_summary["cart_to_transaction_rate"] > 1)
][[
    "itemid",
    "unique_view_visitors",
    "unique_cart_visitors",
    "unique_transaction_visitors",
    "cart_conversion_rate",
    "transaction_conversion_rate",
    "cart_to_transaction_rate"
]]

problem_products.head(20)

,itemid,unique_view_visitors,unique_cart_visitors,unique_transaction_visitors,cart_conversion_rate,transaction_conversion_rate,cart_to_transaction_rate
2999,5951,2,4,0,2.0,0.0,0.0
5033,10040,40,2,3,0.05,0.075,1.5
6430,12836,36,3,4,0.083333,0.111111,1.333333
6713,13398,10,1,2,0.1,0.2,2.0
7678,15283,48,3,4,0.0625,0.083333,1.333333
8884,17724,9,1,2,0.111111,0.222222,2.0
10135,20158,48,1,2,0.020833,0.041667,2.0
10577,21035,10,1,2,0.1,0.2,2.0
11285,22495,58,2,3,0.034483,0.051724,1.5
12346,24509,8,1,2,0.125,0.25,2.0


In [21]:
# Efficient product funnel coverage check

view_visitors = (
    events_enriched.loc[events_enriched["is_view"] == 1]
    .groupby("itemid")["visitorid"]
    .nunique()
)

cart_visitors = (
    events_enriched.loc[events_enriched["is_addtocart"] == 1]
    .groupby("itemid")["visitorid"]
    .nunique()
)

transaction_visitors = (
    events_enriched.loc[events_enriched["is_transaction"] == 1]
    .groupby("itemid")["visitorid"]
    .nunique()
)

product_event_coverage = pd.DataFrame({
    "unique_view_visitors": view_visitors,
    "unique_cart_visitors": cart_visitors,
    "unique_transaction_visitors": transaction_visitors
}).fillna(0).astype(int)

product_event_coverage.head()

,unique_view_visitors,unique_cart_visitors,unique_transaction_visitors
itemid,,,
3,2,0,0
4,3,0,0
6,26,0,0
9,2,0,0
15,13,2,1


In [22]:
# Identify products where downstream activity exceeds upstream activity

coverage_issues = product_event_coverage[
    (product_event_coverage["unique_cart_visitors"] >
     product_event_coverage["unique_view_visitors"])
    |
    (product_event_coverage["unique_transaction_visitors"] >
     product_event_coverage["unique_view_visitors"])
    |
    (product_event_coverage["unique_transaction_visitors"] >
     product_event_coverage["unique_cart_visitors"])
]

coverage_issues.head(20)

,unique_view_visitors,unique_cart_visitors,unique_transaction_visitors
itemid,,,
563,9,0,1
963,12,0,1
1094,0,1,0
2872,0,1,0
2917,32,0,1
5951,2,4,0
6022,0,1,0
6670,0,1,0
7490,0,1,0
